In [2]:
import pyspark.sql as pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [2]:
%%html
<style>
.jp-OutputArea-output pre {
    white-space: pre;
}
</style>

### Создание сессии

In [3]:
spark = (pyspark.SparkSession.builder
    .master('local')
    .appName("My Spark Application")
    .config('spark.executor.memory', '1024m')
    .config("spark.executor.instances", "5")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark

# Датасет IMDb Non-Commercial Datasets
ссылка на скачивание и описание
https://developer.imdb.com/non-commercial-datasets/

# Чтение данных

In [5]:
name_basics_df = spark.read.csv('/user/sandbox/name_basics_tsv', sep='\t', header=True, nullValue=r'\N')

In [6]:
title_akas_df = spark.read.parquet('/user/sandbox/title_akas_par')

In [7]:
title_basics_df = spark.read.parquet('/user/sandbox/title_basics_par')

In [8]:
title_crew_df = spark.read.parquet('/user/sandbox/title_crew_par')

In [9]:
title_episode_df = spark.read.json('/user/sandbox/title_episode_json')

In [10]:
title_principals_df = spark.read.parquet('/user/sandbox/title_principals_par')

In [11]:
ratings_df = spark.read.json('/user/sandbox/title_ratings_json')

# Partitions (считать gz и разбить)

In [13]:
name_basics_df.rdd.getNumPartitions()

6

In [16]:
%time name_basics_df.count()

[Stage 8:=================================================>         (5 + 1) / 6]

CPU times: user 10.1 ms, sys: 488 μs, total: 10.6 ms
Wall time: 5.45 s


14972403

In [14]:
name_basics_gz_df = spark.read.csv('/user/sandbox/name.basics.tsv.gz', sep='\t', header=True, nullValue=r'\N')

In [25]:
name_basics_gz_df.repartition(10).write.csv('/user/sandbox/name.basics.10p', compression='gzip')

In [35]:
df = name_basics_df.repartition(10).withColumn('part', F.spark_partition_id())

In [42]:
df = name_basics_df.withColumn('char', name_basics_df.primaryName.substr(0, 1)).withColumn('part', F.spark_partition_id()).groupby('part').count()

In [36]:
df.groupby('part').count().show()

[Stage 66:===================================================>     (9 + 1) / 10]

+----+-------+
|part|  count|
+----+-------+
|   1|1497240|
|   3|1497240|
|   5|1497240|
|   7|1497240|
|   9|1497241|
|   0|1497240|
|   2|1497240|
|   4|1497241|
|   6|1497241|
|   8|1497240|
+----+-------+



In [15]:
name_basics_gz_df.rdd.getNumPartitions()

1

In [17]:
%time name_basics_gz_df.count()

[Stage 11:>                                                         (0 + 1) / 1]

CPU times: user 4.9 ms, sys: 0 ns, total: 4.9 ms
Wall time: 6.44 s


15106017

In [47]:
title_basics_df.printSchema()

root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)



In [50]:
title_basics_df.schema

StructType([StructField('tconst', StringType(), True), StructField('titleType', StringType(), True), StructField('primaryTitle', StringType(), True), StructField('originalTitle', StringType(), True), StructField('isAdult', StringType(), True), StructField('startYear', StringType(), True), StructField('endYear', StringType(), True), StructField('runtimeMinutes', StringType(), True), StructField('genres', StringType(), True)])

In [59]:
title_basics_df.show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|tt0000048|    short| The Boxing Kangaroo| The Boxing Kangaroo|      0|     1896|   NULL|          NULL|               Short|
|tt0000071|    short|Automobiles Start...|Départ des automo...|      0|     1896|   NULL|          NULL|         Short,Sport|
|tt0000092|    short|Marée montante su...|Marée montante su...|      0|     1896|   NULL|          NULL|   Documentary,Short|
|tt0000099|    short|Place de la Bastille|Place de la Bastille|      0|     1896|   NULL|          NULL|   Documentary,Short|
|tt0000108|    short|Rip Leaving Sleep...|Rip Leaving Sleep...|      0|     1896|   NULL|             1|         Drama

# Работа со сложными структурами данных (получить число фильмов конкретного жанра)

In [58]:
title_basics_df.withColumn('genres_split', F.explode(F.split('genres', ','))).filter("genres_split == 'Documentary'").show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+-----------------+------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|           genres|genres_split|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+-----------------+------------+
|tt0000092|    short|Marée montante su...|Marée montante su...|      0|     1896|   NULL|          NULL|Documentary,Short| Documentary|
|tt0000099|    short|Place de la Bastille|Place de la Bastille|      0|     1896|   NULL|          NULL|Documentary,Short| Documentary|
|tt0000194|    short|Express Train on ...|Express Train on ...|      0|     1898|   NULL|          NULL|Documentary,Short| Documentary|
|tt0000231|    short|Portuguese Railwa...|Portuguese Railwa...|      0|     1896|   NULL|             1|Documentary,Short| Documentary|
|tt0000379|    short|Visita de la escu...|Visita

# UDF

In [75]:
import unicodedata

@F.udf(returnType=T.StructType([T.StructField('first', T.StringType()), T.StructField('second', T.IntegerType())]))
def normalize(x):

    l = list(range(10000000)) // OOM!
    
    return {'first': 'my_string', 'second': len(l)}
    

In [76]:
title_basics_df.select(normalize('originalTitle')['second']).show()

26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_59!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_106!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_32!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_97!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_118!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_127!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_55!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_135!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_190!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_196_0!
26/03/12 16:24:02 WARN BlockManagerMasterEndpoint: No mo

+-------------------------------+
|normalize(originalTitle).second|
+-------------------------------+
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
|                       10000000|
+-------------------------------+
only showing top 20 rows


# Группировки (режиссеры с самым большим числом фильмов)

In [78]:
df = title_basics_df.withColumn('genres_split', F.explode(F.split('genres', ',')))

In [89]:
title_basics_df.show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|tt0000048|    short| The Boxing Kangaroo| The Boxing Kangaroo|      0|     1896|   NULL|          NULL|               Short|
|tt0000071|    short|Automobiles Start...|Départ des automo...|      0|     1896|   NULL|          NULL|         Short,Sport|
|tt0000092|    short|Marée montante su...|Marée montante su...|      0|     1896|   NULL|          NULL|   Documentary,Short|
|tt0000099|    short|Place de la Bastille|Place de la Bastille|      0|     1896|   NULL|          NULL|   Documentary,Short|
|tt0000108|    short|Rip Leaving Sleep...|Rip Leaving Sleep...|      0|     1896|   NULL|             1|         Drama

In [90]:
title_crew_df.show()

+---------+-------------------+---------+
|   tconst|          directors|  writers|
+---------+-------------------+---------+
|tt0000001|          nm0005690|     NULL|
|tt0000010|          nm0525910|     NULL|
|tt0000015|          nm0721526|nm0721526|
|tt0000022|          nm0525910|     NULL|
|tt0000023|          nm0525910|     NULL|
|tt0000031|          nm0525910|     NULL|
|tt0000036|          nm0005690|nm0410331|
|tt0000040|          nm0617588|     NULL|
|tt0000045|          nm0617588|     NULL|
|tt0000049|          nm0010291|     NULL|
|tt0000053|          nm0684607|     NULL|
|tt0000057|          nm0617588|     NULL|
|tt0000082|          nm0005690|     NULL|
|tt0000083|          nm0617588|     NULL|
|tt0000089|nm0525908,nm0698645|     NULL|
|tt0000091|          nm0617588|nm0617588|
|tt0000095|          nm0617588|     NULL|
|tt0000096|          nm0617588|     NULL|
|tt0000097|          nm0617588|     NULL|
|tt0000103|          nm0617588|     NULL|
+---------+-------------------+---

In [95]:
( title_basics_df
    .join(title_crew_df, 'tconst')
    .groupby('directors')
    .count()
    .join(name_basics_df, F.col('directors') == F.col('nconst'))
    .sort('count', ascending=False)
).rdd.getNumPartitions()

[Stage 159:===================================================>   (15 + 1) / 16]

4

In [88]:
df.groupby('tconst').agg(F.collect_list('genres_split')).printSchema()

root
 |-- tconst: string (nullable = true)
 |-- collect_list(genres_split): array (nullable = false)
 |    |-- element: string (containsNull = false)



In [86]:
df.withColumn('startYear', F.col('startYear').cast('int')).groupby('genres_split').agg(F.avg('startYear')).show()

[Stage 114:=============================================>           (4 + 1) / 5]

+------------+------------------+
|genres_split|    avg(startYear)|
+------------+------------------+
|       Crime| 2006.230315526691|
|     Romance|2004.2889547181487|
|    Thriller|2011.9218564684543|
|   Adventure|2005.9211330334683|
|       Drama| 2005.404264623803|
|         War|1996.4301344882608|
| Documentary|2006.7147083910625|
|  Reality-TV|2014.9550830633532|
|      Family|2000.7600707098677|
|     Fantasy| 2004.182702627418|
|   Game-Show| 2004.066594152617|
|       Adult|2012.8157894736842|
|     History|2008.8375491716747|
|     Mystery|2002.8239553680462|
|     Musical|2000.6151011942336|
|   Animation|2006.7744817767225|
|       Music|2000.9570714466297|
|   Film-Noir| 1948.736902050114|
|       Short| 2002.965398675276|
|      Horror|2005.3297156348003|
+------------+------------------+
only showing top 20 rows


# MLlib

In [96]:
import pyspark.ml as ML

In [97]:
df = title_basics_df.join(ratings_df, 'tconst').withColumn('startYear', F.col('startYear').cast('int')).withColumn('averageRating', F.col('averageRating').cast('float')).withColumn('numVotes', F.col('numVotes').cast('int')).cache()

In [99]:
df.write.parquet('/user/sandbox/ml_data', compression='gzip')

In [100]:
df = spark.read.parquet('/user/sandbox/ml_data')

In [101]:
df.show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+
|tt0000339|    short|       La tarentelle|       La tarentelle|      0|     1900|   NULL|          NULL|               Short|          4.0|      13|
|tt0000557|    short| Kathleen Mavourneen| Kathleen Mavourneen|      0|     1906|   NULL|            15| Drama,Romance,Short|          4.6|     162|
|tt0000752|    short| Romance of a Jewess| Romance of a Jewess|      0|     1908|   NULL|            10|         Drama,Short|          5.2|     171|
|tt0001387|    short|A Romance of the ...|A Romance of the ...|      0|     1910|   NULL|            16|Ro

In [102]:
train, test = df.randomSplit([0.8, 0.2], 42)

In [103]:
train.count()

1291547

In [104]:
test.count()

323109

0.20011011633437711

In [106]:
va = ML.feature.VectorAssembler(inputCols=['startYear', 'numVotes'], outputCol='features', handleInvalid='skip')

In [108]:
va.transform(df).show()

[Stage 179:>                                                        (0 + 1) / 1]

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|       features|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------------+
|tt0000339|    short|       La tarentelle|       La tarentelle|      0|     1900|   NULL|          NULL|               Short|          4.0|      13|  [1900.0,13.0]|
|tt0000557|    short| Kathleen Mavourneen| Kathleen Mavourneen|      0|     1906|   NULL|            15| Drama,Romance,Short|          4.6|     162| [1906.0,162.0]|
|tt0000752|    short| Romance of a Jewess| Romance of a Jewess|      0|     1908|   NULL|            10|         Drama,Short|          5.2|     171| [1908.0,171.0]|
|tt0001387

In [116]:
test_features = va.transform(test)

In [113]:
train_features = va.transform(train)

In [46]:
train_features.show()

[Stage 51:>                                                         (0 + 1) / 1]

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|       features|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------------+
|tt0000015|    short|      Around a Cabin| Autour d'une cabine|      0|     1894|   NULL|             2|Animation,Comedy,...|          6.2|    1315|[1894.0,1315.0]|
|tt0000019|    short|    The Clown Barber|    The Clown Barber|      0|     1898|   NULL|          NULL|        Comedy,Short|          4.7|      41|  [1898.0,41.0]|
|tt0000095|    short|The Mysterious Paper|    Le papier protée|      0|     1896|   NULL|          NULL|               Short|          4.6|      43|  [1896.0,43.0]|
|tt0000108

In [111]:
lr = ML.regression.LinearRegression(labelCol='averageRating')

In [115]:
model = lr.fit(train_features)

26/03/12 17:05:10 WARN Instrumentation: [012b71a6] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

In [117]:
model.transform(test_features).show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+--------------+------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|      features|        prediction|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+--------------+------------------+
|tt0000035|    short|Watering the Flowers|          L'arroseur|      0|     1896|   NULL|             1|        Comedy,Short|          5.4|      94| [1896.0,94.0]| 5.728647362219187|
|tt0000054|    short|    A Merry-Go-Round| Les chevaux de bois|      0|     1896|   NULL|          NULL|               Short|          4.8|      43| [1896.0,43.0]| 5.728610731576671|
|tt0000066|    short|Dessinateur: Von ...|Dessinateur: Von ...|      0|     1896|   N

In [119]:
model.params

[Param(parent='LinearRegression_84835aa4be7f', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'),
 Param(parent='LinearRegression_84835aa4be7f', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'),
 Param(parent='LinearRegression_84835aa4be7f', name='epsilon', doc='The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber'),
 Param(parent='LinearRegression_84835aa4be7f', name='featuresCol', doc='features column name.'),
 Param(parent='LinearRegression_84835aa4be7f', name='fitIntercept', doc='whether to fit an intercept term.'),
 Param(parent='LinearRegression_84835aa4be7f', name='labelCol', doc='label column name.'),
 Param(parent='LinearRegression_84835aa4be7f', name='loss', doc='The loss function to be optimized. Supported options: squaredError, huber.'),
 Param(parent='LinearRegression_84835aa4be7f', name='m

In [54]:
model = lr.fit(train_features)

26/03/12 13:37:07 WARN Instrumentation: [93d771be] regParam is zero, which might cause numerical instability and overfitting.
26/03/12 13:37:12 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/12 13:37:12 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
                                                                                

In [63]:
model.transform(va.transform(test)).show()

[Stage 75:==============================================>           (4 + 1) / 5]

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------------+------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|       features|        prediction|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------------+------------------+
|tt0000058|    short|Cortège de tzar a...|Cortège de tzar a...|      0|     1896|   NULL|          NULL|   Documentary,Short|          3.9|      42|  [1896.0,42.0]| 5.728237573122934|
|tt0000145|    short|           En classe|           En classe|      0|     1897|   NULL|          NULL|        Comedy,Short|          3.4|      21|  [1897.0,21.0]| 5.739671579572407|
|tt0000174|    short|Výstavní párkar a...|Výstavní párkar a...|      0|     1898

In [66]:
df.show()

[Stage 83:=====================================================>(199 + 1) / 200]

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+
|tt0000658|    short|The Puppet's Nigh...|Le cauchemar de F...|      0|     1908|   NULL|             2|     Animation,Short|          6.4|     332|
|tt0001732|    short|The Lighthouse Ke...|The Lighthouse Ke...|      0|     1911|   NULL|            10|         Drama,Short|          5.9|      40|
|tt0002253|    short|          Home Folks|          Home Folks|      0|     1912|   NULL|            17|         Drama,Short|          5.3|      31|
|tt0002473|    short|    The Sands of Dee|    The Sands of Dee|      0|     1912|   NULL|            17|  

In [120]:
df_with_genres = df.withColumn('genre', F.split('genres', ',')[0])

In [121]:
df_with_genres.show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+-----------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|      genre|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+-----------+
|tt0000339|    short|       La tarentelle|       La tarentelle|      0|     1900|   NULL|          NULL|               Short|          4.0|      13|      Short|
|tt0000557|    short| Kathleen Mavourneen| Kathleen Mavourneen|      0|     1906|   NULL|            15| Drama,Romance,Short|          4.6|     162|      Drama|
|tt0000752|    short| Romance of a Jewess| Romance of a Jewess|      0|     1908|   NULL|            10|         Drama,Short|          5.2|     171|      Drama|
|tt0001387|    short|A Romance of 

In [125]:
si = ML.feature.StringIndexer(inputCol='genre', outputCol='genre_index').setHandleInvalid('skip')

In [128]:
df_with_genres_indices = si.fit(df_with_genres).transform(df_with_genres)

In [131]:
df_with_genres_indices.select('genre').drop_duplicates().count()

27

In [132]:
with_onehot = ML.feature.OneHotEncoder(inputCol='genre_index', outputCol='onehot_genre').fit(df_with_genres_indices)

In [133]:
va = ML.feature.VectorAssembler(inputCols=['startYear', 'numVotes', 'onehot_genre'], outputCol='features', handleInvalid='skip')

In [135]:
va.transform(with_onehot.transform(df_with_genres_indices)).show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+-----------+-----------+---------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|      genre|genre_index|   onehot_genre|            features|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+-----------+-----------+---------------+--------------------+
|tt0000339|    short|       La tarentelle|       La tarentelle|      0|     1900|   NULL|          NULL|               Short|          4.0|      13|      Short|        8.0| (26,[8],[1.0])|(28,[0,1,10],[190...|
|tt0000557|    short| Kathleen Mavourneen| Kathleen Mavourneen|      0|     1906|   NULL|            15| Drama,Romance,Short|          4.6|     162|      Drama|

In [136]:
gbtr = ML.regression.GBTRegressor(labelCol='averageRating')

In [139]:
pipeline = ML.pipeline.Pipeline(stages=[si, with_onehot, va, gbtr])

In [140]:
pipeline

Pipeline_21749a9854a0

In [141]:
train, test = df_with_genres.randomSplit([0.8, 0.2], 42)

In [142]:
model = pipeline.fit(train)

26/03/12 17:25:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [149]:
test_eval = model.transform(test)

In [147]:
rmse = ML.evaluation.RegressionEvaluator(labelCol='averageRating')

In [150]:
rmse.evaluate(test_eval)

1.3449141560177853

In [74]:
model = si.fit(df_with_genres).setHandleInvalid('skip')

In [80]:
with_genre_index = model.transform(df_with_genres)

In [85]:
with_onehot = ML.feature.OneHotEncoder(inputCol='genre_index', outputCol='onehot_genre').fit(with_genre_index).transform(with_genre_index)

In [87]:
df_with_genres.show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|    genre|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------+
|tt0000658|    short|The Puppet's Nigh...|Le cauchemar de F...|      0|     1908|   NULL|             2|     Animation,Short|          6.4|     332|Animation|
|tt0001732|    short|The Lighthouse Ke...|The Lighthouse Ke...|      0|     1911|   NULL|            10|         Drama,Short|          5.9|      40|    Drama|
|tt0002253|    short|          Home Folks|          Home Folks|      0|     1912|   NULL|            17|         Drama,Short|          5.3|      31|    Drama|
|tt0002473|    short|    The Sands of Dee|    

In [94]:
si = ML.feature.StringIndexer(inputCol='genre', outputCol='genre_index', handleInvalid='skip')

In [100]:
onehotEncoder = ML.feature.OneHotEncoder(inputCol='genre_index', outputCol='onehot_genre')

In [96]:
va = ML.feature.VectorAssembler(inputCols=['startYear', 'numVotes', 'onehot_genre'], outputCol='features', handleInvalid='skip')

In [97]:
lr = ML.regression.LinearRegression(labelCol='averageRating')

In [108]:
gbtr = ML.regression.GBTRegressor(labelCol='averageRating')

In [109]:
pipeline = ML.Pipeline(stages=[si, onehotEncoder, va, gbtr])

In [110]:
train, test = df_with_genres.randomSplit([0.8, 0.2], 42)

In [111]:
pipelineModel = pipeline.fit(train)

In [105]:
pipelineModel.transform(df_with_genres).show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------+-----------+---------------+--------------------+------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|averageRating|numVotes|    genre|genre_index|   onehot_genre|            features|        prediction|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+-------------+--------+---------+-----------+---------------+--------------------+------------------+
|tt0000658|    short|The Puppet's Nigh...|Le cauchemar de F...|      0|     1908|   NULL|             2|     Animation,Short|          6.4|     332|Animation|        6.0| (26,[6],[1.0])|(28,[0,1,8],[1908...| 5.907498490719828|
|tt0001732|    short|The Lighthouse Ke...|The Lighthouse Ke...|      0|     1911|   NULL|   

In [112]:
from pyspark.ml.evaluation import RegressionEvaluator